In [50]:
# general libraries
import numpy as np
import matplotlib.pyplot as plt

# main libraries
import keras
import tensorflow as tf
from keras.layers import Conv2D, MaxPool2D, Dropout, Flatten, Dense, BatchNormalization, GlobalAvgPool2D
from keras.models import Sequential
from keras.callbacks import ModelCheckpoint, EarlyStopping

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from keras.models import load_model

In [51]:
model = Sequential()
model.add(Conv2D(filters = 16, kernel_size = (3,3), activation="relu", input_shape=(224,224,3) ))

model.add(Conv2D(filters = 32, kernel_size = (3,3), activation="relu" ))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 64, kernel_size = (3,3), activation="relu" ))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(filters = 128, kernel_size = (3,3), activation="relu" ))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Dropout(rate = .25))

model.add(Flatten())
model.add(Dense(units = 64, activation= "relu"))
model.add(Dropout(rate = 0.25))
model.add(Dense( units = 1, activation= "sigmoid"))

# model.compile(optimizer="adam",loss=keras.losses.BinaryCrossentropy(),metrics=["accuracy"])
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision")
    ]
)

model.summary()



/home/coco/Learning/personal/Brain-Tumor-Detection-Model-using-Keras/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 222, 222, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 220, 220, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 110, 110, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 108, 108, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │     5,537,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,635,361 (21.50 MB)

 Trainable params: 5,635,361 (21.50 MB)

 Non-trainable params: 0 (0.00 B)

In [52]:
def imagePreaparation1(path):

    # ip : Path
    # op : processed images

    imageData = ImageDataGenerator(zoom_range = .2,         #data augmentation
                                   shear_range = .2,
                                   rescale = 1/255,
                                   horizontal_flip = True)

    image = imageData.flow_from_directory(directory = path,
                                          target_size = (224,224),
                                          batch_size = 32,
                                          class_mode = "binary")

    return image


In [53]:
def imagePreaparation2(path):

    # ip : Path
    # op : processed images

    imageData = ImageDataGenerator(rescale = 1/255)

    image = imageData.flow_from_directory(directory = path,
                                          target_size = (224,224),
                                          batch_size = 32,
                                          class_mode = "binary")

    return image


In [54]:
# calling and preaparing
tainingDataPath = "data/Training"
trainingData = imagePreaparation1(tainingDataPath)
testingDataPath = "data/Testing"
testingData = imagePreaparation2(testingDataPath)
validationDataPath = "data/Validation"
validationData = imagePreaparation2(validationDataPath)



Found 5712 images belonging to 2 classes.
Found 666 images belonging to 2 classes.
Found 645 images belonging to 2 classes.


In [55]:
# early stopping and model checking
# early stopping
# es = EarlyStopping(monitor="val_accuracy",
#                    min_delta=.01,
#                    patience=3,
#                    verbose=1,
#                    mode= 'auto')

es = EarlyStopping(
    monitor="val_recall",
    min_delta=0.02,
    patience=4,
    verbose=1,
    mode="max"
)



In [56]:
# # model check point
# mc = ModelCheckpoint(monitor="val_accuracy",
#                      filepath='./bestModel.keras',
#                      verbose=1,
#                      save_best_only=True,
#                      mode='auto')

# cd = [es,mc]

mc = ModelCheckpoint(
    monitor="val_recall",
    filepath="./bestModel.keras",
    save_best_only=True,
    mode="max",
    verbose=1
)

cd = [es, mc]


In [57]:
# Model Training
# hs = model.fit(trainingData,
#                steps_per_epoch= 8,
#                epochs = 30,
#                verbose = 1,
#                validation_data = validationData,
#                validation_steps = 16,
#                callbacks = cd)
class_weight = {
    0: 1.0,   # notumor
    1: 3.0    # tumor (more important)
}


hs = model.fit(
    trainingData,
    steps_per_epoch=8,
    epochs=30,
    validation_data=validationData,
    validation_steps=16,
    callbacks=cd,
    class_weight=class_weight
)




Epoch 1/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 550ms/step - loss: 1.6243 - precision: 0.7445 - recall: 0.7817
Epoch 1: val_recall improved from None to 1.00000, saving model to ./bestModel.keras

Epoch 1: finished saving model to ./bestModel.keras
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 869ms/step - loss: 1.4445 - precision: 0.7222 - recall: 0.9185 - val_loss: 1.0148 - val_precision: 0.6953 - val_recall: 1.0000
Epoch 2/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 548ms/step - loss: 1.1238 - precision: 0.7241 - recall: 1.0000
Epoch 2: val_recall did not improve from 1.00000
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 824ms/step - loss: 1.0472 - precision: 0.7227 - recall: 1.0000 - val_loss: 0.7334 - val_precision: 0.6816 - val_recall: 1.0000
Epoch 3/30
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 539ms/step - loss: 0.8728 - precision: 0.7544 - recall: 1.0000
Epoch 3: val_recall did not improve from 1.00000
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 823ms/step - loss: 0.8232 - precision: 0.7578 - recall: 1.0000 - val_loss: 0.7891 - val_precision: 0.6875 - val_recall: 

In [58]:
#model interpreatation

h = hs.history
h.keys()

dict_keys(['loss', 'precision', 'recall', 'val_loss', 'val_precision', 'val_recall'])

In [59]:
plt.plot(h["accuracy"])
plt.plot(h["val_accuracy"])
plt.title("Model Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epoch")
plt.legend(["Training","Validation"],loc="upper left")
plt.show()

plt.plot(h["loss"])
plt.plot(h["val_loss"])
plt.title("Model Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Training","Validation"],loc="upper left")
plt.show()

KeyError: 'accuracy'

In [60]:
model = load_model("./bestModel.keras")

loss, acc = model.evaluate(testingData)
print("Accuracy:", acc)
print("Loss:", loss)

21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 125ms/step - loss: 1.0419 - precision: 0.6877 - recall: 1.0000


ValueError: too many values to unpack (expected 2)

In [61]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = testingData.classes

y_pred_prob = model.predict(testingData)   # shape: (N, 1)
y_pred = (y_pred_prob >= 0.5).astype(int).ravel()

print(confusion_matrix(y_true, y_pred))
print(classification_report(
    y_true,
    y_pred,
    target_names=testingData.class_indices.keys()
))


21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 120ms/step
[[  0 208]
 [  0 458]]
              precision    recall  f1-score   support

     notumor       0.00      0.00      0.00       208
       tumor       0.69      1.00      0.81       458

    accuracy                           0.69       666
   macro avg       0.34      0.50      0.41       666
weighted avg       0.47      0.69      0.56       666



/home/coco/Learning/personal/Brain-Tumor-Detection-Model-using-Keras/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/coco/Learning/personal/Brain-Tumor-Detection-Model-using-Keras/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/coco/Learning/personal/Brain-Tumor-Detection-Model-using-Keras/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no 